# 01 · Generate Synthetic Corpus  ·  ground-truth WRITER
Deterministically generates ~1.5–3k legacy claim notes across ~300 claims and, **in the same pass**, writes the sealed ground-truth manifest with the exact char offset of every planted mention. Then seals sha256 of every raw note. After this notebook the raw corpus is read-only for the rest of the pipeline.

In [ ]:
# --- bootstrap: make the src package importable from any working dir ---
import sys
from pathlib import Path
p = Path.cwd().resolve()
while not (p / 'config' / '00_config.py').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
print('project root:', p)


In [ ]:
from src import corpus_gen
from src.hashing import write_hashes, verify_hashes
summary = corpus_gen.generate_corpus()
print('corpus summary:', summary)


In [ ]:
# seal hashes (written once) and immediately verify
hashes = write_hashes(overwrite=True)
report = verify_hashes('post-generation')
print('sealed', len(hashes), 'raw files; integrity ok =', report['ok'])


In [ ]:
# peek at one raw note and validate a few planted offsets are byte-accurate
import json
from src.settings import Paths
man = json.loads(Paths.manifest_json.read_text())
doc = man['documents'][0]['doc_id']
print('--- sample note', doc, '---')
print((Paths.raw_notes / f'{doc}.txt').read_text()[:700])
ok = 0
for pl in man['placements'][:2000]:
    t = (Paths.raw_notes / f"{pl['doc_id']}.txt").read_text()
    ok += (t[pl['char_start']:pl['char_end']] == pl['surface_variant'])
print('planted-offset fidelity (first 2000):', ok, '/ 2000')
print('entities:', len(man['entities']), '| placements:', len(man['placements']),
      '| non_entities:', len(man['non_entities']))
